# 文から関係の三つ組を抽出する (REBEL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_semantics/relation_extraction.ipynb)

NLP クックブック 16-07 ／ 意味解析

文章から「(夏目漱石, 職業, 小説家)」のような (主語, 関係, 目的語) の三つ組を取り出して、知識グラフを作りたい。

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

name = "Babelscape/rebel-large"   # Wikipedia と Wikidata の対応から学習した、関係抽出専用の系列変換モデル (BART)
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name).eval()

def parse(decoded):
    """モデルの出力 '<triplet> 主語 <subj> 目的語 <obj> 関係 <subj> 目的語 <obj> 関係 <triplet> ...' を三つ組のリストにする"""
    triples, head, tail, rel, state = [], "", "", "", None
    for tok in decoded.replace("<s>", "").replace("</s>", "").replace("<pad>", "").split():
        if tok == "<triplet>":
            if head and tail and rel:
                triples.append((head.strip(), rel.strip(), tail.strip()))
            head, tail, rel, state = "", "", "", "head"
        elif tok == "<subj>":
            if head and tail and rel:
                triples.append((head.strip(), rel.strip(), tail.strip()))
            tail, rel, state = "", "", "tail"
        elif tok == "<obj>":
            rel, state = "", "rel"
        elif state == "head":
            head += " " + tok
        elif state == "tail":
            tail += " " + tok
        elif state == "rel":
            rel += " " + tok
    if head and tail and rel:
        triples.append((head.strip(), rel.strip(), tail.strip()))
    return triples

def extract(text, num_beams=4, num_return_sequences=3):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        out = model.generate(**enc, max_length=200, num_beams=num_beams, num_return_sequences=num_return_sequences)
    triples = []
    for seq in tokenizer.batch_decode(out, skip_special_tokens=False):   # 上位の候補を合わせて再現率を上げる
        triples += [t for t in parse(seq) if t not in triples]
    return triples

texts = ["Natsume Soseki was a Japanese novelist born in Tokyo in 1867. He wrote the novel Kokoro.",
         "Toyota Motor Corporation is a Japanese automobile manufacturer headquartered in Toyota City, Aichi. It was founded by Kiichiro Toyoda in 1937."]
for text in texts:
    print(text)
    for t in extract(text):
        print("   ", t)

Natsume Soseki was a Japanese novelist born in Tokyo in 1867. He wrote the novel Kokoro.
    ('Natsume Soseki', 'date of birth', '1867')
    ('Kokoro', 'author', 'Natsume Soseki')
    ('Natsume Soseki', 'place of birth', 'Tokyo')
Toyota Motor Corporation is a Japanese automobile manufacturer headquartered in Toyota City, Aichi. It was founded by Kiichiro Toyoda in 1937.
    ('Toyota Motor Corporation', 'country', 'Japan')
    ('Toyota Motor Corporation', 'product or material produced', 'automobile')
    ('Toyota Motor Corporation', 'headquarters location', 'Toyota City')
    ('Toyota Motor Corporation', 'founded by', 'Kiichiro Toyoda')
    ('Toyota Motor Corporation', 'inception', '1937')
    ('Japan', 'contains administrative territorial entity', 'Aichi')
    ('Toyota City', 'country', 'Japan')
    ('Toyota City', 'located in the administrative territorial entity', 'Aichi')
    ('Aichi', 'country', 'Japan')
    ('Aichi', 'contains administrative territorial entity', 'Toyota City')
   

In [2]:
import networkx as nx

G = nx.DiGraph()
for text in texts:
    for head, rel, tail in extract(text):
        G.add_edge(head, tail, relation=rel)
print(G.number_of_nodes(), "ノード", G.number_of_edges(), "エッジ")
print([(h, t, d["relation"]) for h, t, d in G.edges(data=True) if "Toyota" in h][:5])

11 ノード 13 エッジ
[('Toyota Motor Corporation', 'Japan', 'country'), ('Toyota Motor Corporation', 'automobile', 'product or material produced'), ('Toyota Motor Corporation', 'Toyota City', 'headquarters location'), ('Toyota Motor Corporation', 'Kiichiro Toyoda', 'founded by'), ('Toyota Motor Corporation', '1937', 'inception')]


- 関係の種類は Wikidata のプロパティ (約 220 種類) に限られる。それ以外の関係や、ドメイン固有の関係は取れない。
- 文に書かれていない関係を「知識」から出力してしまうことがある。抽出した主語・目的語が原文に出現するかで検証するとよい。
- 多言語版の mREBEL もあるが、日本語の精度は英語に及ばない。日本語では、係り受けと固有表現を組み合わせた規則 (構文解析の章の三つ組抽出) や、この章の最初の KWJA による述語項構造解析が現実的な選択肢になる。